# BP5 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Root-Cause & Driver Analytics**

## Why this notebook exists, and what it honestly scopes
Master Execution Plan Section 5.1 / Section 7 define BP5 as: *"hypothesis testing, regression, and
explainability (SHAP) to identify associations between product/issue/process fields and outcomes;
every finding labeled as association, with a reminder that association is not causation."* This is
BP5's very first notebook, opening Sprint 4 (Section 23) alongside BP4. It is the project's first
BP that is neither a supervised classifier (BP1/BP2/BP3) nor descriptive aggregation (BP4) —
association/driver testing is a genuinely new methodology for this suite, and this notebook grounds
its scope against the real, already-profiled CFPB extract and BP3's own already real-run-confirmed
Gate 1 policy (`notebooks/bp3_complaint_escalation_prediction/artifacts/policy.json`) before any
Gate 3 hypothesis-testing/regression/SHAP work is built.

## Data source: the same real CFPB outcome-field family BP3 already draws on, not a new source
Master Plan Section 6 (Integration Architecture) states the data flow explicitly: *"CFPB outcome
fields [...] into BP3/BP5"* — plural, naming the same real outcome-field family for both BPs. This
notebook honors that literally: BP5 reads the identical real, raw CFPB extract BP1–BP4 already read
(`data/raw/cfpb_complaints.csv`), re-verifies its real 15-column schema live (Section 4 below, same
check BP2 Gate 1 first established and BP3/BP4 Gate 1 each reused), and draws its outcome fields from
the same `Company response to consumer` / `Timely response?` field family BP2/BP3 Gate 1 already
profiled — never a different or invented data source. **No BANKING77 integration**: the Master Plan's
BP table marks BP5 `Integrates BANKING77? = NO`, matching BP3; the Gold-layer `common_taxonomy_bucket`
column (BP1/BP2's own Gate 2 work) is not read in this notebook at all.

## Two real, distinct outcome fields — a disclosed design decision, not a data limitation
Section 6's Integration Architecture wording is plural ("CFPB outcome fields"), and this notebook
takes that literally rather than collapsing BP5 onto a single reused target:

- **`outcome_1_intervention_required`** — reused **verbatim** from BP3 Gate 1's own real, already
  confirmed target definition on `Company response to consumer` (HYPER: not reinvented). Binary:
  1 if `"Closed with monetary relief"`; 0 if `"Closed with explanation"` or `"Closed with
  non-monetary relief"`. `"In progress"`, `"Untimely response"`, and null-response rows are excluded
  from the trainable set, identically to BP3 Gate 1's own reasoning. BP3 Gate 1's own real run
  (REAL-RUN CONFIRMED 2026-09-23 per the Evidence Ledger) recorded, on the real full 1,048,575-row
  extract: `n_intervention_required` = 10,511, `n_no_intervention_required` = 804,942,
  `n_trainable_total` = 815,453, `n_excluded_total` = 233,122 (231,856 in progress + 1,264 untimely
  response + 2 null), positive-class ratio 0.0129 — cited here from that already-real-run artifact
  as established context, and **independently re-computed live again in Section 6 below** rather
  than trusted blindly (this project's own stated Gate 1 prerequisite discipline).
- **`outcome_2_timely_response_failure`** — new to BP5: the second real CFPB outcome field the
  Master Plan's own plural wording anticipates. Binary: 1 if the real `Timely response?` value is
  `"No"` (CFPB's own independent timeliness judgment); 0 if `"Yes"`. Computed over the full real
  population, independent of `Company response to consumer`'s value; null `Timely response?` rows
  (BP3 Gate 1's own real run found zero) excluded from the trainable set. `Timely response?` has
  never been used as a primary target/outcome by BP1–BP4 — BP2 Gate 1 and BP3 Gate 1 each only
  live-enumerated its 2 real distinct values (`"Yes"`, `"No"`) for feature-barring purposes, never
  modeled it as an outcome in its own right.

**Why two outcomes rather than one:** collapsing BP5 onto `intervention_required` alone would make
BP5's scope nearly redundant with BP3 (same field, same partition, same rows). The Master Plan's own
plural "CFPB outcome fields" wording for BP5's data flow, plus the existence of a second real,
independent, never-yet-modeled CFPB outcome field (`Timely response?`), together support treating
BP5 as testing real product/issue/process drivers against **two** genuinely distinct outcomes rather
than one duplicated target. Whether these two outcome fields are themselves independent, or
correlated (which would need disclosing, not hiding), is **not assumed either way** — Section 6 below
live-computes their real cross-tabulation (how much `Company response to consumer == "Untimely
response"` overlaps with `Timely response? == "No"`, and how much `outcome_1`'s positive class
overlaps with `outcome_2`'s) rather than assuming they are independent signals or the same signal
under two names.

## What "product/issue/process" driver fields honestly means here
The Master Plan's own BP5 wording is "product/issue/process fields," not an open-ended feature set.
This notebook defines BP5's Gate 1 candidate driver fields explicitly, mirroring BP3 Gate 1's own
feature-variable-candidates list (HYPER, same real structured fields, same frequency-encoding note
for `Company`):
- **Product/issue fields**: `Product`, `Sub-product`, `Issue`, `Sub-issue` — CFPB's own real
  product/issue taxonomy, used as-is.
- **Process fields**: `Submitted via` (the real intake channel) and `Company` (the real responding
  firm, frequency-encoded at Gate 2/3 given its real high cardinality — HYPER, reusing BP2/BP3's own
  pattern in `src/features/`, never one-hot encoded).
- **Secondary control field**: `State` — kept only as a geographic control dimension for Gate 3/4
  modeling, **never presented as a primary named "product/issue/process" driver** per the Master
  Plan's own BP5 wording. This is an explicit scoping choice, not a claim that geography carries no
  real association.

**Deliberately excluded from the Gate 1 candidate driver set** (see `leakage_rules` below for the
full reasoning): `Company public response` (real, live-verified high null rate and outcome-adjacent
content — Section 6), `Date received` / `Date sent to company` (barred as a conservative default,
mirroring BP3 Gate 1), `Tags` (demographic-adjacent, re-verified live in Section 5), `Complaint ID`
and `ZIP code` (identifier / quasi-identifier, BP2/BP3/BP4's own `BARRED_COLUMNS` precedent).

## BP5's own leakage-analog: outcome-echo contamination, not classic train/test leakage
Gate 1's own exit criterion (Section 8: "No target leakage possible by construction") is reframed
here for an association-testing BP with **two** real outcome fields instead of one supervised
target. The risk this notebook guards against is not classic train/test leakage (BP5 trains no
model at Gate 1) but **outcome-echo contamination**: treating one real CFPB outcome field as a
"driver" of the other, when both are themselves outcome-adjacent measurements of the same
underlying complaint-handling event. Concretely: `Company response to consumer` (and
`outcome_1_intervention_required` itself) is barred as a candidate driver for `outcome_2`, and
`Timely response?` (and `outcome_2_timely_response_failure` itself) is barred as a candidate driver
for `outcome_1` — symmetric bars, both grounded in Section 6's live overlap check rather than
assumed. This mirrors BP3 Gate 1's own already-disclosed reasoning for barring `Timely response?`
from BP3's feature set: BP2 Gate 2 found 1,963 real rows where `Timely response?` disagrees with
`Company response to consumer`, so the true association-vs-leakage risk between these two fields
was already flagged as unverified before BP5 existed.

## Methodology intended for Gate 3/4 (policy-level only — nothing is tested in this Gate 1 notebook)
Per the Master Plan's own BP5 methodology (Section 5.1/7) and named-metric rule (Section 14: "no
self-invented scoring rubrics anywhere in this suite"), Gate 1 records **what** will be tested, not
**how** — no hypothesis test, regression, or SHAP computation runs in this notebook:
- **Hypothesis testing**: chi-square test of independence (categorical driver × binary outcome)
  with Cramér's V effect size, per real candidate driver field, per outcome.
- **Regression**: logistic regression (statsmodels/scikit-learn, both already listed in this
  project's `requirements.txt`) per real candidate driver field, coefficients reported with
  confidence intervals, per Master Plan Section 10.2's own BP4–BP5 SMART-objective wording ("real
  chi-square/regression coefficients with confidence intervals").
- **Explainability**: SHAP (also already listed in `requirements.txt`) applied to the Gate 3
  champion model on a representative sample only, never the full corpus, per Master Plan Section 14.
- Both outcomes are tested against the same real candidate driver fields as **two separate**
  association analyses — never combined into one joint target.

## Association is not causation — a structural policy field, not only a markdown comment
The Master Plan states this as a hard rule for BP5 twice: Section 5.1/7's own methodology line
("every finding labeled as association, with a reminder that association is not causation") and
Section 9's UDAAP mapping for BP5 specifically ("root-cause findings framed as potential risk
indicators only when statistically supported"). This notebook writes
`target_definition.association_not_causation_disclaimer` into `policy.json` and the config YAML as
an explicit, structural field carried into every later BP5 gate — not left to a one-time comment
here.

## Compliance touchpoints — verified against the full Master Plan document, not the excerpt alone
- **UDAAP** (Section 9): BP5 is explicitly named, alongside BP2 and BP6, as a BP whose
  root-cause/friction/GenAI findings must be "framed as potential risk indicators only when
  statistically supported" — operationalized via the disclaimer field above.
- **ECOA / Regulation B is NOT a BP5 compliance touchpoint.** Section 9's Regulatory Frameworks
  table maps ECOA/Reg B to *"BP1, BP2, BP3, BP7, wherever any demographic-adjacent field could
  appear"* — BP5 is not in that list, confirmed by reading the full Master Plan document (not just
  the task excerpt). This notebook states that plainly as Not Applicable for BP5's own compliance
  touchpoint, rather than silently reusing BP3's ECOA/Reg B statement. `Tags` is still barred from
  BP5's candidate driver set anyway, as a conservative scope decision (Section 5, re-verified live,
  not assumed) — HYPER-consistent with BP4 Gate 1's own precedent of barring `Tags` even where
  ECOA/Reg B does not map to the BP in question.
- **GLBA/GDPR-aligned data-minimization & purpose-limitation statement** — Gate 1's own Section 8
  compliance touchpoint for every BP: BP5 reads only the real CFPB structured fields already in
  scope for BP1–BP4 (no new external data source), for the single stated purpose of association
  testing, and attempts no re-identification (no consumer identifier exists in this real extract,
  re-confirmed live in Section 4).
- **CFPB supervisory & complaint-handling standards** (Section 9: "Governs: All 8 BPs") — BP5's
  candidate driver fields are CFPB's own real product/issue/sub-issue schema, used as-is; no
  fabricated outcome is introduced.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it
  on your own machine, and the real, live-checked results below become this project's Gate 1 policy
  record for BP5.
- **Zero-fabrication** (Section 12.1): every check below runs against the real file in `data/raw/`.
  No category label, count, or distinct-value string in this notebook is asserted from memory —
  BP3 Gate 1's own already-real-run numbers are cited above with their source stated explicitly, and
  every number this notebook itself needs is independently re-computed live in Section 5/6, never
  copied from BP3's artifact.
- **WARP**: `configure_performance()` first. The same real 1,048,575-row CFPB file BP1–BP4 each
  read at Gate 1, loaded via `pl.scan_csv` (lazy), exactly as BP1–BP4 Gate 1 each do.
- **HYPER**: reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (real column dtypes, no
  re-derivation), `src/utils/bp1_config_sync.py` (generic marker-based config read/write, already
  reused unmodified by BP2/BP3/BP4), and BP2 Gate 1's own ID-like-keyword check for "no persistent
  customer/consumer identifier."
- **Idempotent**: re-running this notebook overwrites `configs/bp5_root_cause_driver_analytics.yaml`
  (front matter only) and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Outcome definitions (Business Understanding, Master Plan Section 5.1/7, BP5)
- **`outcome_1_intervention_required`** — binary, reused verbatim from BP3 Gate 1 on
  `Company response to consumer` (see above for the full definition and exclusions).
- **`outcome_2_timely_response_failure`** — binary, new to BP5, on `Timely response?` (see above).
- **Candidate driver fields**: `Product`, `Sub-product`, `Issue`, `Sub-issue` (product/issue),
  `Submitted via`, `Company` (process), `State` (secondary control only) — see above for the full
  exclusion reasoning on every other real CFPB field.

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp5_root_cause_driver_analytics.yaml` — `target_definition`, `leakage_rules`,
  `assumptions`, `status` written to the front-matter section (existing gate blocks, if any,
  preserved verbatim)
- `notebooks/bp5_root_cause_driver_analytics/artifacts/policy.json` — the Section 8 Gate 1 output
  artifact, with the live category-enumeration, both outcomes' class balances, their real overlap,
  and the candidate driver-field profile all embedded

## Prerequisites
`01_data_acquisition_profiling.ipynb` and BP3's own Gate 1 (which first defined
`intervention_required` on this identical field) should have been real-run at least once. This
notebook re-verifies everything it needs independently rather than trusting either artifact blindly.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A failing outcome-echo check in particular
must never be worked around — if a barred field's real overlap with an outcome makes it look like a
shortcut into that outcome, that is a real methodological risk (an "association" finding that is
actually just one outcome field echoing another) and must be fixed in the candidate driver set, not
in this check.

In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp5_root_cause_driver_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp5_root_cause_driver_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"

# ============================================================
# SECTION 4: Structural checks - real CFPB schema re-verified live (not asserted from
# RAW_DATA_MANIFEST.md's or any prior BP's own documentation alone), plus the same
# "no persistent customer/consumer identifier" check BP2 Gate 1 first established and
# BP3/BP4 Gate 1 each reused (HYPER).
# ============================================================
cfpb_columns = list(CFPB_DTYPES.keys())
EXPECTED_CFPB_COLUMNS = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company public response",
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID",
]
ID_LIKE_KEYWORDS = ("customer", "consumer id", "person", "account number", "ssn", "email", "phone")
suspected_customer_id_columns = [c for c in cfpb_columns if any(kw in c.lower() for kw in ID_LIKE_KEYWORDS)]
print(f"[OK] Real CFPB columns ({len(cfpb_columns)}): {cfpb_columns}")
print(
    f"[OK] Columns matching customer/consumer-identifier keywords: {suspected_customer_id_columns or 'NONE'}"
)

# ============================================================
# SECTION 5: Live enumeration of the real candidate outcome/compliance fields - actual label
# strings and real counts fetched live here, never asserted from memory or copied from any
# prior BP's already-documented values (BP3/BP4 Gate 1's own zero-fabrication precedent).
# ============================================================
cfpb_lazy = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)

company_response_counts = (
    cfpb_lazy.group_by("Company response to consumer")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)
timely_response_counts = (
    cfpb_lazy.group_by("Timely response?").agg(pl.len().alias("n")).sort("n", descending=True).collect()
)
tags_counts = cfpb_lazy.group_by("Tags").agg(pl.len().alias("n")).sort("n", descending=True).collect()

total_rows = cfpb_lazy.select(pl.len()).collect().item()

print(f"[OK] Real CFPB row count (live): {total_rows:,}")
print("[OK] Live 'Company response to consumer' distinct values + real counts:")
print(company_response_counts)
print("[OK] Live 'Timely response?' distinct values + real counts:")
print(timely_response_counts)
print("[OK] Live 'Tags' distinct values + real counts (conservative re-check, see markdown):")
print(tags_counts)

DEMOGRAPHIC_ADJACENT_KEYWORDS = ("servicemember", "older american", "veteran")
demographic_adjacent_tags_found = [
    v
    for v in tags_counts["Tags"].to_list()
    if v is not None and any(kw in str(v).lower() for kw in DEMOGRAPHIC_ADJACENT_KEYWORDS)
]
print(f"[OK] Demographic-adjacent 'Tags' values found live: {demographic_adjacent_tags_found or 'NONE'}")

# ============================================================
# SECTION 6: Live computation of BP5's two real, distinct outcome fields, their real class
# balances, real candidate-driver-field cardinality/nulls, and the real overlap between the two
# outcome fields (grounds the "two real outcome fields, not one duplicated" design choice in
# actual data rather than assumption).
# ============================================================
response_col = "Company response to consumer"
timely_col = "Timely response?"

# --- Outcome 1: intervention_required - reused verbatim from BP3 Gate 1's own real target
# definition on this identical field (see markdown). Recomputed live here, independently, per
# this project's zero-fabrication rule - never copied from BP3's own already-real-run artifact.
n_intervention_required = (
    cfpb_lazy.filter(pl.col(response_col) == "Closed with monetary relief").select(pl.len()).collect().item()
)
n_no_intervention_required = (
    cfpb_lazy.filter(
        pl.col(response_col).is_in(["Closed with explanation", "Closed with non-monetary relief"])
    )
    .select(pl.len())
    .collect()
    .item()
)
n_excluded_in_progress = (
    cfpb_lazy.filter(pl.col(response_col) == "In progress").select(pl.len()).collect().item()
)
n_excluded_untimely_response = (
    cfpb_lazy.filter(pl.col(response_col) == "Untimely response").select(pl.len()).collect().item()
)
n_excluded_null_response = cfpb_lazy.filter(pl.col(response_col).is_null()).select(pl.len()).collect().item()
n_trainable_outcome1 = n_intervention_required + n_no_intervention_required
n_excluded_outcome1_total = n_excluded_in_progress + n_excluded_untimely_response + n_excluded_null_response

outcome_1_class_balance = {
    "n_intervention_required": int(n_intervention_required),
    "n_no_intervention_required": int(n_no_intervention_required),
    "n_trainable_total": int(n_trainable_outcome1),
    "n_excluded_in_progress": int(n_excluded_in_progress),
    "n_excluded_untimely_response": int(n_excluded_untimely_response),
    "n_excluded_null_response": int(n_excluded_null_response),
    "n_excluded_total": int(n_excluded_outcome1_total),
    "positive_class_ratio_of_trainable": (
        round(n_intervention_required / n_trainable_outcome1, 4) if n_trainable_outcome1 else None
    ),
}
print(f"[OK] Live BP5 outcome_1 (intervention_required) class balance: {outcome_1_class_balance}")

# --- Outcome 2: timely_response_failure - a second, real CFPB outcome field, newly used here
# (never used as a primary target by BP1-BP4), independent of 'Company response to consumer'.
n_timely_yes = cfpb_lazy.filter(pl.col(timely_col) == "Yes").select(pl.len()).collect().item()
n_timely_no = cfpb_lazy.filter(pl.col(timely_col) == "No").select(pl.len()).collect().item()
n_timely_null = cfpb_lazy.filter(pl.col(timely_col).is_null()).select(pl.len()).collect().item()
n_trainable_outcome2 = n_timely_yes + n_timely_no

outcome_2_class_balance = {
    "n_timely_response_failure": int(n_timely_no),
    "n_timely_response_ok": int(n_timely_yes),
    "n_trainable_total": int(n_trainable_outcome2),
    "n_excluded_null_timely_response": int(n_timely_null),
    "positive_class_ratio_of_trainable": (
        round(n_timely_no / n_trainable_outcome2, 4) if n_trainable_outcome2 else None
    ),
}
print(f"[OK] Live BP5 outcome_2 (timely_response_failure) class balance: {outcome_2_class_balance}")

# --- Real overlap between the two outcome fields - grounds the "two distinct real outcome
# fields, not two names for one signal" design choice in an actual live cross-tabulation,
# mirroring BP3 Gate 1's own real, disclosed overlap check against BP2's target on this same
# underlying data.
n_untimely_response_and_timely_no = (
    cfpb_lazy.filter((pl.col(response_col) == "Untimely response") & (pl.col(timely_col) == "No"))
    .select(pl.len())
    .collect()
    .item()
)
n_untimely_response_and_timely_yes = (
    cfpb_lazy.filter((pl.col(response_col) == "Untimely response") & (pl.col(timely_col) == "Yes"))
    .select(pl.len())
    .collect()
    .item()
)
n_intervention_required_and_timely_no = (
    cfpb_lazy.filter((pl.col(response_col) == "Closed with monetary relief") & (pl.col(timely_col) == "No"))
    .select(pl.len())
    .collect()
    .item()
)
outcome_overlap_live_check = {
    "n_untimely_response_rows": int(n_excluded_untimely_response),
    "n_untimely_response_and_timely_no": int(n_untimely_response_and_timely_no),
    "n_untimely_response_and_timely_yes": int(n_untimely_response_and_timely_yes),
    "pct_untimely_response_rows_also_timely_no": (
        round(n_untimely_response_and_timely_no / n_excluded_untimely_response, 4)
        if n_excluded_untimely_response
        else None
    ),
    "n_timely_no_rows": int(n_timely_no),
    "pct_timely_no_rows_also_untimely_response": (
        round(n_untimely_response_and_timely_no / n_timely_no, 4) if n_timely_no else None
    ),
    "n_intervention_required_and_timely_no": int(n_intervention_required_and_timely_no),
    "pct_intervention_required_rows_also_timely_no": (
        round(n_intervention_required_and_timely_no / n_intervention_required, 4)
        if n_intervention_required
        else None
    ),
}
print("[OK] Live outcome-field overlap check (Company response to consumer x Timely response?):")
print(outcome_overlap_live_check)

# --- Real candidate driver-field cardinality/null profile - Product, Sub-product, Issue,
# Sub-issue, Company, Submitted via (product/issue/process fields per Master Plan Section 5.1/7)
# plus State as a secondary/control geographic dimension only (see markdown for why State is not
# a primary named driver).
CANDIDATE_DRIVER_FIELDS = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
    "State",
]
candidate_driver_field_profile = {}
for _col in CANDIDATE_DRIVER_FIELDS:
    n_unique = cfpb_lazy.select(pl.col(_col).n_unique()).collect().item()
    n_null = cfpb_lazy.filter(pl.col(_col).is_null()).select(pl.len()).collect().item()
    candidate_driver_field_profile[_col] = {
        "n_unique": int(n_unique),
        "n_null": int(n_null),
        "pct_null": round(n_null / total_rows, 4) if total_rows else None,
    }
print(f"[OK] Live candidate driver-field cardinality/null profile: {candidate_driver_field_profile}")

# --- Real null profile of 'Company public response' - documents (does not use as a feature)
# why it is excluded from BP5's Gate 1 candidate driver list (see markdown/leakage_rules).
n_null_company_public_response = (
    cfpb_lazy.filter(pl.col("Company public response").is_null()).select(pl.len()).collect().item()
)
pct_null_company_public_response = (
    round(n_null_company_public_response / total_rows, 4) if total_rows else None
)
print(
    f"[OK] Real 'Company public response' null count (live, excluded field, informational only): "
    f"{n_null_company_public_response:,} ({pct_null_company_public_response:.2%})"
)

# ============================================================
# SECTION 7: Assemble the Gate 1 policy (target definition, leakage rules, ASSUMPTIONs)
# ============================================================
policy = {
    "bp_id": "bp5",
    "bp_name": "bp5_root_cause_driver_analytics",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_definition": {
        "no_single_supervised_target_two_real_outcome_fields": True,
        "scoping_note": "Master Plan Section 6 (Integration Architecture) names 'CFPB outcome "
        "fields' (plural) feeding BP3/BP5 - BP5 draws on that same real CFPB "
        "outcome-field family BP3's Gate 1 already defined and used, not a "
        "different data source. BP5 tests real product/issue/process fields for "
        "statistical association against TWO real, distinct outcome fields "
        "(below), not one - grounded live in Section 6's outcome-field overlap "
        "check, not assumed redundant or assumed independent.",
        "outcome_1_intervention_required": {
            "description": "Reused verbatim from BP3 Gate 1's own real target definition on the "
            "identical 'Company response to consumer' field (HYPER: not "
            "reinvented). Binary: 1 if 'Closed with monetary relief'; 0 if "
            "'Closed with explanation' or 'Closed with non-monetary relief'. "
            "'In progress', 'Untimely response', and null-response rows "
            "excluded from the trainable set - identical exclusion reasoning "
            "to BP3 Gate 1 (see this notebook's markdown cell).",
            "source_field": "Company response to consumer",
            "reused_from": "bp3_complaint_escalation_prediction Gate 1 (identical definition, "
            "independently re-verified live in Section 6 below, never copied "
            "from BP3's own already-real-run artifact).",
        },
        "outcome_2_timely_response_failure": {
            "description": "New to BP5 - the second real CFPB outcome field the Master Plan's "
            "own plural 'CFPB outcome fields' wording anticipates. Binary: 1 if "
            "the real 'Timely response?' value is 'No' (CFPB's own timeliness "
            "judgment); 0 if 'Yes'. Computed over the full real population "
            "(independent of 'Company response to consumer''s value); null "
            "'Timely response?' rows excluded from the trainable set.",
            "source_field": "Timely response?",
            "distinct_from_outcome_1_by_construction": "'Timely response?' is CFPB's own "
            "independent timeliness assessment, not a value of 'Company "
            "response to consumer' - the real overlap between this outcome "
            "and 'Company response to consumer' == 'Untimely response' is "
            "live-verified in Section 6 (outcome_field_overlap_live_check "
            "below) rather than assumed to be either identical or unrelated.",
        },
        "candidate_driver_fields": {
            "product_issue_fields": ["Product", "Sub-product", "Issue", "Sub-issue"],
            "process_fields": ["Submitted via", "Company"],
            "secondary_control_field": "State - kept only as a geographic control dimension, "
            "never presented as a named 'product/issue/process' driver per the "
            "Master Plan's own BP5 wording.",
            "company_encoding_note": "'Company' is real high-cardinality and will be "
            "frequency-encoded at Gate 2/3 (HYPER-reusing BP2/BP3's own "
            "pattern in src/features/) - not one-hot encoded.",
            "excluded_from_candidate_set": {
                "Company public response": "Excluded at Gate 1 - real null rate live-verified "
                "in Section 6 (BP2 Gate 1's own prior finding was "
                "~54% null; re-verified live and independently here, not "
                "carried over), and its content is the company's own "
                "public statement ABOUT the complaint's outcome, an "
                "outcome-adjacent field BP2 Gate 3 already flagged for an "
                "explicit leakage test before use. Deferred to a later "
                "gate, same posture as BP3 Gate 1's own omission of this "
                "field from its candidate feature list.",
                "Date received / Date sent to company": "Barred as a conservative default - see "
                "leakage_rules.",
                "Tags": "Barred - demographic-adjacent, re-verified live in Section 5. See "
                "leakage_rules and compliance_touchpoint.",
                "Complaint ID / ZIP code": "Barred as a pure identifier and a fine-grained "
                "quasi-identifier respectively - matches BP2/BP3/BP4's own "
                "BARRED_COLUMNS precedent (HYPER-consistent).",
            },
        },
        "methodology_policy": {
            "note": "Policy-level only - defines WHAT Gate 3/4 will test, not HOW. No hypothesis "
            "test, regression, or SHAP run in this Gate 1 notebook.",
            "hypothesis_testing": "Chi-square test of independence (categorical driver x binary "
            "outcome) with Cramer's V effect size, per Master Plan Section 14's own "
            "named-metric rule (no self-invented scoring rubric).",
            "regression": "Logistic regression (statsmodels/scikit-learn) per real candidate "
            "driver field, coefficients reported with confidence intervals, per Master "
            "Plan Section 10.2's own BP4-BP5 SMART objective wording.",
            "explainability_shap": "SHAP applied to the Gate 3 champion model on a representative "
            "sample only, never the full corpus, per Master Plan Section 14.",
            "two_outcomes_tested_separately": "Both outcome_1 and outcome_2 are tested against "
            "the same real candidate driver fields as two separate association "
            "analyses, never combined into a single joint target.",
        },
        "association_not_causation_disclaimer": "Every BP5 finding, at every later gate, is "
        "reported as a statistical ASSOCIATION between a real candidate driver field and a "
        "real outcome field - never as a causal claim. This is a structural policy field, not "
        "only a markdown comment, per Master Plan Section 5.1/7's own explicit BP5 methodology "
        "rule ('every finding labeled as association, with a reminder that association is not "
        "causation') and Section 9's UDAAP mapping for BP5 ('root-cause findings framed as "
        "potential risk indicators only when statistically supported'). No BP5 deliverable at "
        "any gate may claim or imply that a driver field CAUSES an outcome.",
    },
    "leakage_rules": [
        "'Company response to consumer' defines outcome_1 (intervention_required) and must "
        "never also be used as a candidate driver field for outcome_1 - it is the outcome "
        "itself, not traditional train/test leakage.",
        "'Timely response?' defines outcome_2 (timely_response_failure) and must never also be "
        "used as a candidate driver field for outcome_2, for the same reason.",
        "'Company response to consumer' (and outcome_1 itself) is barred as a candidate driver "
        "field for outcome_2 (timely_response_failure) - an outcome-echo risk, not classic "
        "leakage: Section 6's live overlap check quantifies how much 'Untimely response' (one "
        "of 'Company response to consumer''s own values) coincides with 'Timely response?' == "
        "'No' on the real data, and treating one real outcome field as a 'driver' of a second, "
        "overlapping real outcome field would not be a genuine product/issue/process "
        "association finding.",
        "'Timely response?' (and outcome_2 itself) is barred as a candidate driver field for "
        "outcome_1 (intervention_required), symmetrically, and as a conservative default "
        "consistent with BP3 Gate 1's own bar on this same field for the same target - BP2 "
        "Gate 2 already found 1,963 real rows where 'Timely response?' disagrees with 'Company "
        "response to consumer', so its true association-vs-leakage risk for outcome_1 is "
        "unverified. Deferred to Gate 3 for an explicit test before any relaxation.",
        "'Date received' and 'Date sent to company' are barred from the candidate driver set "
        "for BOTH outcomes as a conservative default - a computed response-time duration was "
        "found to leak the timeliness-derived half of BP2's own target on this identical data "
        "(BP2 Gate 3 finding), and 'Timely response?' (outcome_2 here) is itself a timeliness "
        "judgment on the same two dates. Deferred to Gate 3 for an explicit test, the same "
        "posture BP3 Gate 1 took on these identical fields.",
        "'Tags' is barred from the candidate driver set for both outcomes - live-re-checked in "
        "Section 5 rather than trusting any prior Gate 1's now-superseded 'no protected-class "
        "field' claim, and found (or re-confirmed found, matching BP2 Gate 3's and BP3 Gate 1's "
        "real findings) to contain demographic-adjacent values.",
        "'Company public response' is excluded from BP5's Gate 1 candidate driver set - see "
        "target_definition.candidate_driver_fields.excluded_from_candidate_set for the real, "
        "live-verified null-rate and outcome-adjacency reasoning.",
        "'Complaint ID' and 'ZIP code' are barred as a pure identifier and a fine-grained "
        "quasi-identifier respectively - matches BP2/BP3/BP4's own BARRED_COLUMNS precedent "
        "(HYPER-consistent).",
        "No BANKING77 data is used for BP5 at all (Master Plan BP table: Integrates BANKING77? "
        "= NO) - not loaded in this notebook; the Gold-layer common_taxonomy_bucket column is "
        "not read here.",
        "Every BP5 finding at every later gate is reported as an ASSOCIATION, never a causal "
        "claim - see target_definition.association_not_causation_disclaimer.",
    ],
    "assumptions": [
        "BP5 tests statistical association against two real, distinct CFPB outcome fields "
        "(intervention_required, reused from BP3 Gate 1's own definition, and "
        "timely_response_failure, newly defined here) rather than a single target - an "
        "explicit, disclosed design choice grounded in Master Plan Section 6's own plural "
        "'CFPB outcome fields' wording and in Section 6's live overlap check below, open to "
        "Gate 3 review if the real class balance or overlap makes either outcome untenable.",
        "'Company public response' is excluded from BP5's Gate 1 candidate driver set given its "
        "real, live-verified null rate and its outcome-adjacent content (the company's own "
        "public statement about the complaint) - not proven necessary, open to Gate 2/3 review.",
        "'Timely response?', 'Date received', and 'Date sent to company' are barred from the "
        "candidate driver set for BOTH outcomes at Gate 1 as a conservative default, not yet "
        "proven necessary - deferred to Gate 3 for an explicit association/leakage test before "
        "any relaxation, the same practice BP3 Gate 1 established for these same fields.",
        "'State' is kept only as a secondary geographic control field, never presented as a "
        "primary named 'product/issue/process' driver per the Master Plan's own BP5 wording - "
        "an explicit scoping choice, not a claim that geography has no real association.",
        "No CFPB field in this real 15-column extract is treated as demographic/protected-class "
        "by name; 'Tags' is barred anyway as a conservative default given its live-verified "
        "demographic-adjacent values, consistent with BP3/BP4 Gate 1's own precedent, even "
        "though ECOA/Reg B does not map to BP5 per Master Plan Section 9's own mapping (BP1, "
        "BP2, BP3, BP7 only - re-verified against the full Master Plan document, not the "
        "excerpt alone).",
        "src/utils/bp1_config_sync.py is reused unmodified for BP5's own config file - already "
        "fully generic, parameterized by config_path, no BP1-specific logic (already reused "
        "unmodified by BP2, BP3, and BP4).",
    ],
    "compliance_touchpoint": {
        "requirement": "UDAAP - Unfair, Deceptive, or Abusive Acts or Practices (Master Plan "
        "Section 9; BP5 is one of the three BPs this section explicitly maps to UDAAP, alongside "
        "BP2 and BP6)",
        "statement": "Per Master Plan Section 9's own UDAAP row, BP5's root-cause/driver "
        "findings are framed as potential risk indicators only when statistically supported - "
        "operationalized here as the structural association_not_causation_disclaimer field "
        "above, carried into every later BP5 gate's reporting, never softened to a causal claim "
        "at any point.",
        "ecoa_reg_b_not_applicable": "Master Plan Section 9's Regulatory Frameworks table maps "
        "ECOA / Regulation B to BP1, BP2, BP3, and BP7 only - BP5 is explicitly NOT listed "
        "(re-verified against the full Master Plan document text, not the excerpt alone). "
        "Stated honestly as Not Applicable for BP5's own compliance touchpoint, rather than "
        "silently reusing BP3's ECOA/Reg B statement. 'Tags' is still barred from BP5's "
        "candidate driver set anyway, as a conservative scope decision (see leakage_rules), "
        "HYPER-consistent with BP3/BP4 Gate 1's own precedent of barring 'Tags' even where "
        "ECOA/Reg B does not map to the BP in question (BP4).",
        "data_minimization_purpose_limitation": "Gate 1's own Section 8 compliance touchpoint "
        "(GLBA/GDPR-aligned) for every BP: BP5 reads only the real CFPB structured fields "
        "already in scope for BP1-BP4 (no new external data source), uses each field only for "
        "the specific association-testing purpose defined above, and does not re-identify or "
        "attempt to recover any consumer identity (none exists in this real extract, re-"
        "confirmed live in Section 4).",
    },
    "live_checks": {
        "cfpb_row_count": total_rows,
        "cfpb_columns": cfpb_columns,
        "cfpb_columns_match_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
        "suspected_customer_id_columns": suspected_customer_id_columns,
        "company_response_to_consumer_distribution": company_response_counts.to_dicts(),
        "timely_response_distribution": timely_response_counts.to_dicts(),
        "tags_distribution": tags_counts.to_dicts(),
        "demographic_adjacent_tags_found": demographic_adjacent_tags_found,
        "outcome_1_intervention_required_class_balance": outcome_1_class_balance,
        "outcome_2_timely_response_failure_class_balance": outcome_2_class_balance,
        "outcome_field_overlap_live_check": outcome_overlap_live_check,
        "candidate_driver_field_profile": candidate_driver_field_profile,
        "company_public_response_null_profile": {
            "n_null": n_null_company_public_response,
            "pct_null": pct_null_company_public_response,
        },
    },
}

# ============================================================
# SECTION 8: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2, default=str)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp5_config_path = CONFIGS_DIR / "bp5_root_cause_driver_analytics.yaml"

# BP5 reuses BP1's marker-based config-sync helpers as-is (src/utils/bp1_config_sync.py) -
# already generic, already reused unmodified by BP2, BP3, and BP4. Gate 1 here owns only the
# front-matter section below; every later gate's block (once written) is preserved verbatim
# regardless of position or order. See LESSONS_LEARNED_APPLIED.md #20 for the real incident
# this pattern was built to prevent.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp5_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((2, "Gate 2"), (3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp5_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1/BP2/BP3/BP4 - fully generic, parameterized by config_path); Gates 2-5 each own
# exactly one marker-delimited block appended after it via write_gate_block() - do not hand-edit
# either section, re-run the owning notebook instead.
bp_id: "bp5"
bp_name: "bp5_root_cause_driver_analytics"
status: "gate1_confirmed{_status_suffix}"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete
target_definition:
  no_single_supervised_target_two_real_outcome_fields: true
  scoping_note: "Master Plan Section 6 names 'CFPB outcome fields' (plural) feeding BP3/BP5 -
    BP5 draws on that same real outcome-field family BP3 Gate 1 already defined, not a
    different data source. Tests real product/issue/process fields for association against TWO
    real outcome fields, not one."
  outcome_1_intervention_required: "Reused verbatim from BP3 Gate 1's own real definition on
    'Company response to consumer'. Binary: 1 if 'Closed with monetary relief'; 0 if 'Closed
    with explanation' or 'Closed with non-monetary relief'. 'In progress', 'Untimely response',
    and null-response rows excluded from the trainable set."
  outcome_2_timely_response_failure: "New to BP5. Binary: 1 if real 'Timely response?' == 'No';
    0 if 'Yes'. Computed over the full real population, independent of 'Company response to
    consumer'. Null 'Timely response?' rows excluded from the trainable set."
  candidate_driver_fields:
    product_issue_fields: ["Product", "Sub-product", "Issue", "Sub-issue"]
    process_fields: ["Submitted via", "Company"]
    secondary_control_field: "State - geographic control only, never a named primary driver."
  association_not_causation_disclaimer: "Every BP5 finding at every later gate is reported as a
    statistical ASSOCIATION between a real candidate driver field and a real outcome field -
    never a causal claim (Master Plan Section 5.1/7 and Section 9 UDAAP mapping for BP5)."
leakage_rules:
  - "'Company response to consumer' defines outcome_1 and must never be a candidate driver for
     outcome_1; 'Timely response?' defines outcome_2 and must never be a candidate driver for
     outcome_2."
  - "'Company response to consumer' (outcome_1) barred as a candidate driver for outcome_2, and
     'Timely response?' (outcome_2) barred as a candidate driver for outcome_1 - an outcome-echo
     risk, live-quantified in Section 6's overlap check, not classic leakage."
  - "'Date received' and 'Date sent to company' barred from the candidate driver set for BOTH
     outcomes as a conservative default - a computed response-time duration leaked BP2's own
     target on this identical data (BP2 Gate 3 finding), and outcome_2 is itself a timeliness
     judgment on these same two dates. Deferred to Gate 3 for an explicit test."
  - "'Tags' barred for both outcomes - live-re-checked (not assumed) and found to contain
     demographic-adjacent values, matching BP2/BP3/BP4's own real findings."
  - "'Company public response' excluded from the candidate driver set - real, live-verified
     null rate and outcome-adjacent content (see policy.json for the exact figures)."
  - "'Complaint ID' and 'ZIP code' barred as identifier / quasi-identifier, matching
     BP2/BP3/BP4's own BARRED_COLUMNS precedent."
  - "No BANKING77 data used at all for BP5 (Master Plan BP table: Integrates BANKING77? = NO)."
  - "Every BP5 finding at every later gate is reported as an ASSOCIATION, never a causal claim."
assumptions:
  - "BP5 tests association against two real, distinct CFPB outcome fields (intervention_required,
     reused from BP3 Gate 1, and timely_response_failure, new here) rather than a single target -
     grounded in Master Plan Section 6's plural 'CFPB outcome fields' wording and in Section 6's
     live overlap check, open to Gate 3 review."
  - "'Company public response' excluded from BP5's Gate 1 candidate driver set given its real,
     live-verified null rate and outcome-adjacent content - not proven necessary, open to
     Gate 2/3 review."
  - "'Timely response?', 'Date received', and 'Date sent to company' barred from the candidate
     driver set for BOTH outcomes at Gate 1 as a conservative default, deferred to Gate 3 for an
     explicit association/leakage test."
  - "'State' kept only as a secondary geographic control field, never a primary named driver."
  - "ECOA/Reg B does not map to BP5 per Master Plan Section 9's own mapping (BP1, BP2, BP3, BP7
     only, re-verified against the full document) - 'Tags' still barred anyway as a conservative
     default given its live-verified demographic-adjacent values."
  - "src/utils/bp1_config_sync.py reused unmodified for BP5's own config file - already fully
     generic, no BP1-specific logic."
random_state: 42
"""
write_front_matter(bp5_config_path, bp5_config_text)
print(f"[SAVED] {bp5_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
_all_candidate_and_barred_fields_disjoint = not (
    set(CANDIDATE_DRIVER_FIELDS)
    & {response_col, timely_col, "Tags", "Complaint ID", "ZIP code", "Company public response",
       "Date received", "Date sent to company"}
)

checks = {
    "cfpb_schema_matches_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
    "no_customer_identifier_column_in_cfpb_schema": len(suspected_customer_id_columns) == 0,
    "company_response_to_consumer_enumerated": company_response_counts.height > 0,
    "timely_response_enumerated": timely_response_counts.height > 0,
    "tags_enumerated": tags_counts.height > 0,
    "outcome_1_row_accounting_matches_total": (
        n_trainable_outcome1 + n_excluded_outcome1_total
    )
    == total_rows,
    "outcome_2_row_accounting_matches_total": (n_trainable_outcome2 + n_timely_null) == total_rows,
    "at_least_one_row_per_outcome_1_class": (
        n_intervention_required > 0 and n_no_intervention_required > 0
    ),
    "at_least_one_row_per_outcome_2_class": n_timely_yes > 0 and n_timely_no > 0,
    "no_barred_field_in_candidate_driver_set": _all_candidate_and_barred_fields_disjoint,
    "candidate_driver_fields_all_profiled": (
        len(candidate_driver_field_profile) == len(CANDIDATE_DRIVER_FIELDS)
    ),
    "policy_json_written": policy_json_path.exists(),
    "bp5_config_yaml_written": bp5_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP5 Gate 1 complete - two real, distinct CFPB outcome fields defined "
    "(intervention_required, reused from BP3; timely_response_failure, new here), real overlap "
    "between them live-quantified, real candidate product/issue/process driver fields profiled, "
    "outcome-echo leakage rules recorded, and the association-not-causation policy stated "
    "structurally. Proceed to BP5 Gate 2 (Data Verification & Feature/Taxonomy Engineering) next."
)